# 1. Modelagem — tempo mínimo cinemático

Problema alinhado à seção 4.2 de Massaro et al., *Robotics* 2023: levar um
manipulador planar de dois elos de \(\mathbf{q}(0)=\mathbf{0}\) a
\(\mathbf{q}(t_f)=(1,-0{,}5)\) rad, em **repouso–repouso**, no menor \(t_f\),
sem colidir com um disco no plano de trabalho.

**O que entra.** Cinemática direta, limites de velocidade e de aceleração,
um obstáculo circular, três pontos monitorados no segundo elo.

**O que fica de fora (simplificação em relação ao paper).** Dinâmica de
Lagrange, jerk, colocação LGR/GPOPS-II. A transcrição é aceleração constante
por intervalo com integração exata da cadeia \(\ddot q=a\).


In [ ]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
for cand in [here, here.parent, here / "T2", here.parent / "T2"]:
    if (cand / "src" / "parametros.py").exists():
        if str(cand) not in sys.path:
            sys.path.insert(0, str(cand))
        break
from src.ambiente import preparar
T2 = preparar()
print("T2 =", T2)


## OCP contínuo (Bolza)

Estado \(\mathbf{x}=\{q_1,q_2,v_{q_1},v_{q_2}\}^\mathsf{T}\), controle
\(\mathbf{u}=\{a_{q_1},a_{q_2}\}^\mathsf{T}\),

\[
\dot q_i=v_{q_i},\qquad \dot v_{q_i}=a_{q_i},\qquad
J=\int_0^{t_f} 1\,dt=t_f.
\]

Caminho: \(|\dot q_1|\le 3\), \(|\dot q_2|\le 8\) rad/s, \(|a_i|\le 18\) rad/s²,
e para \(\zeta\in\{1/3,2/3,1\}\)

\[
(x_P(\mathbf{q},\zeta)-x_0)^2+(y_P-y_0)^2\ge R^2,
\]

com \(x_P=l_1\cos q_1+\zeta l_2\cos(q_1+q_2)\) (eq. 6 do paper).


In [ ]:
from src.parametros import P, ZETA, N_Z, N_EQ, N_INEQ
from src.cinematica import ponto_elo2, juntas_elo2
from src.transcricao import (
    chute_desvio, chute_reta, construir_chutes, avaliar, checar_jacobiana, n_decisao
)

print("l1, l2 =", P.l1, P.l2)
print("q0, qf =", P.q0, P.qf)
print("v_max, a_max =", P.v_max, P.a_max)
print("obstáculo =", (P.x0_obs, P.y0_obs, P.R_obs), "zeta =", ZETA)
print("K =", P.K, "dimensão de z =", n_decisao(), "igualdades =", N_EQ, "desigualdades =", N_INEQ)


## PNL discreta

Decisão \(\mathbf{z}=(t_f,\,a_{1,0},\ldots,a_{1,K-1},\,a_{2,0},\ldots,a_{2,K-1})\in\mathbb{R}^{21}\),
\(K=10\), \(h=t_f/K\):

\[
v_{i,k+1}=v_{i,k}+h a_{i,k},\qquad
q_{i,k+1}=q_{i,k}+h v_{i,k}+\tfrac12 h^2 a_{i,k},
\]

com \(q_{i,0}\) e \(v_{i,0}=0\) fixos. Igualdades: \(q_{i,K}=q_{i,f}\), \(v_{i,K}=0\).
Obstáculo nos nós **e** nos pontos médios dos intervalos.


In [ ]:
z_reta = chute_reta()
z_desvio = chute_desvio()
a_reta = avaliar(z_reta)
a_desvio = avaliar(z_desvio)

def relatorio(nome, a):
    tr = a["trajetoria"]
    print(f"{nome:18s} tf={tr.tf:.3f}s  eq={a['violacao_eq']:.2e}  ineq={a['violacao_ineq']:.2e}  "
          f"viavel={a['viavel']}  colide={a['colide']}  qf={tr.q[:,-1]}  vf={tr.v[:,-1]}")

relatorio("reta_juntas", a_reta)
relatorio("desvio_esquerda", a_desvio)
print()
print("cinco chutes da bateria:")
for nome, z in construir_chutes():
    relatorio(nome, avaliar(z))


In [ ]:
err_eq, err_ineq = checar_jacobiana(z_desvio)
print(f"erro relativo da Jacobiana  igualdades={err_eq:.2e}  desigualdades={err_ineq:.2e}")
print("A Jacobiana analítica é a que os pontos interiores recebem.")
